# 第五章：PyTorch 入门

## 从 NumPy 到 PyTorch——现代深度学习框架的核心抽象

第四章我们用纯 NumPy 实现了 MLP 和反向传播，现在引入 PyTorch——它将**自动求导、GPU 加速、
模块化设计**融入工作流，让研究者专注于模型设计而非梯度计算。

本章假设已安装 PyTorch：`pip install torch torchvision`


## 4.1 Tensor：PyTorch 的核心数据结构

Tensor 是 PyTorch 对多维数组的抽象，等价于 NumPy 的 `ndarray` + 自动求导 + GPU 支持。

### Tensor vs ndarray

| 特性 | NumPy ndarray | PyTorch Tensor |
|------|--------------|----------------|
| GPU 加速 | ❌ | ✅ `.to('cuda')` |
| 自动求导 | ❌ | ✅ `requires_grad=True` |
| 动态计算图 | ❌ | ✅ Define-by-Run |
| API 风格 | NumPy | 与 NumPy 高度兼容 |


In [ ]:
import torch
import numpy as np

# === 从数据创建 Tensor ===
# 从列表
a = torch.tensor([1, 2, 3, 4, 5])
print(f"从列表: {a}, dtype={a.dtype}")

# 从 NumPy
b = torch.from_numpy(np.array([1.0, 2.0, 3.0]))
print(f"从NumPy: {b}")

# 特殊矩阵
zeros = torch.zeros(3, 4)       # 全零
ones = torch.ones(2, 3)          # 全一
rand = torch.randn(3, 3)         # 标准正态分布
eye = torch.eye(3)               # 单位矩阵
arange = torch.arange(0, 10, 2)  # 类似 range

print(f"zeros shape: {zeros.shape}")
print(f"rand: \n{rand}")

# 类型转换
c = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"float32: {c}, dtype={c.dtype}")
c_long = c.long()
print(f"long: {c_long}, dtype={c_long.dtype}")


In [ ]:
# === Tensor 运算 ===
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# 逐元素运算
print("x + y =", x + y)
print("x * y =", x * y)  # 逐元素乘（不是矩阵乘！）
print("x ** 2 =", x ** 2)

# 矩阵运算
print("x @ y =\n", x @ y)           # 矩阵乘法
print("x.matmul(y) =\n", x.matmul(y))

# 形状操作
z = torch.arange(12).reshape(3, 4)
print(f"reshape:\n{z}")
print(f"转置:\n{z.T}")
print(f"view:\n{z.view(4, 3)}")
print(f"permute:\n{z.permute(1, 0)}")  # 维度重排

# 归约操作
print(f"sum: {z.sum()}")
print(f"mean dim=0: {z.mean(dim=0)}")  # 按行平均
print(f"max dim=1: {z.max(dim=1)}")    # 按列找最大

# 索引与切片（与 NumPy 相同）
print(f"第一行: {z[0]}")
print(f"前两行: {z[:2]}")
print(f"第二列: {z[:, 1]}")


In [ ]:
# === GPU 支持 ===
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"当前 GPU: {torch.cuda.get_device_name(0)}")
    
    # 数据转移到 GPU
    x_gpu = x.to('cuda')
    print(f"x 在 GPU 上: {x_gpu.device}")
    
    # 在 GPU 上运算
    y_gpu = y.to('cuda')
    result = x_gpu @ y_gpu
    print(f"GPU 计算结果:\n{result}")
    
    # 转回 CPU
    result_cpu = result.cpu()
    print(f"回到 CPU: {result_cpu.device}")
else:
    print("未检测到 GPU，使用 CPU 模式")


## 4.2 Autograd：自动求导引擎

PyTorch 的 **autograd** 系统自动记录所有 Tensor 操作，构建**动态计算图**，然后自动计算
任意 Tensor 对任意参数的梯度。这是 PyTorch 区别于 NumPy 的核心特性。

### 工作原理

1. 设置 `requires_grad=True` 标记需要梯度的 Tensor
2. 前向计算时，autograd 记录所有操作
3. 调用 `.backward()` 自动计算梯度
4. 梯度存储在 `.grad` 属性中

### 一个简单的例子

$$
f(x) = x^3 + 2x^2 + x \quad \Rightarrow \quad f'(x) = 3x^2 + 4x + 1
$$

在 $x=2$ 处：$f'(2) = 12 + 8 + 1 = 21$


In [ ]:
# === Autograd 基础 ===
x = torch.tensor(2.0, requires_grad=True)

# 前向计算
y = x ** 3 + 2 * x ** 2 + x  # f(x) = x^3 + 2x^2 + x
print(f"f(2) = {y}")

# 反向传播
y.backward()
print(f"f'(2) = {x.grad}")  # 应该是 21
print(f"手动计算: 3*2^2 + 4*2 + 1 = {3*4 + 4*2 + 1}")

# 复杂例子：多变量函数
a = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
b = torch.tensor([4.0, 5.0, 6.0], requires_grad=True)

f = (a * b).sum()  # f = sum(a_i * b_i)
f.backward()
print(f"\na.grad = {a.grad}")  # ∂f/∂a_i = b_i
print(f"b.grad = {b.grad}")    # ∂f/∂b_i = a_i


In [ ]:
# === 梯度下降 with Autograd ===
# 目标：找到 f(x) = (x-3)^2 的最小值（显然 x=3）
x = torch.tensor(0.0, requires_grad=True)
lr = 0.1
history = []

for step in range(30):
    loss = (x - 3) ** 2  # f(x) = (x-3)^2
    
    loss.backward()
    
    with torch.no_grad():  # 更新时不记录梯度
        x -= lr * x.grad
        x.grad.zero_()     # 清空梯度（否则会累加！）
    
    history.append((step, x.item(), loss.item()))

for s, xv, lv in history[::5]:
    print(f"Step {s:2d}: x={xv:.4f}, loss={lv:.6f}")

print(f"\n最终: x={x.item():.6f} (真实答案: 3.0)")

# 画收敛曲线
import matplotlib.pyplot as plt
steps, xs, losses = zip(*history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(steps, xs); ax1.axhline(3, color='r', linestyle='--')
ax1.set_title('x'); ax1.set_xlabel('Step'); ax1.grid(True)
ax2.plot(steps, losses); ax2.set_title('Loss')
ax2.set_xlabel('Step'); ax2.grid(True)
plt.tight_layout()
plt.savefig('/home/yisan/ai-learning/notebooks/autograd_gd.png', dpi=100, bbox_inches='tight')
plt.show()


### 梯度清零的重要性

每次调用 `.backward()` 时，梯度是**累加**到 `.grad` 上的。忘记 `.zero_()` 是初学者最常犯的错误：

```python
# ❌ 错误：梯度会累加，导致参数更新方向错误
for step in range(100):
    loss = model(x)
    loss.backward()
    optimizer.step()  # grads = grads_old + grads_new

# ✅ 正确：
for step in range(100):
    optimizer.zero_grad()  # 先清零
    loss = model(x)
    loss.backward()
    optimizer.step()
```


## 4.3 nn.Module：构建神经网络的标准范式

`nn.Module` 是所有神经网络层的基类。它封装了参数（weights, biases）和前向传播逻辑。

### 核心组件

| 组件 | 用途 |
|------|------|
| `nn.Linear(in, out)` | 全连接层：$y = Wx + b$ |
| `nn.Conv2d(in, out, kernel)` | 2D 卷积层 |
| `nn.ReLU()` | ReLU 激活函数 |
| `nn.Sigmoid()` | Sigmoid 激活函数 |
| `nn.BatchNorm2d(n)` | 批归一化 |
| `nn.Dropout(p)` | Dropout 正则化 |
| `nn.Sequential(...)` | 顺序容器 |


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# === 方式 1: Sequential（简单堆叠）===
model_seq = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
print("Sequential model:")
print(model_seq)

# 查看参数
total_params = sum(p.numel() for p in model_seq.parameters())
trainable_params = sum(p.numel() for p in model_seq.parameters() if p.requires_grad)
print(f"总参数: {total_params:,} (可训练: {trainable_params:,})")

# 前向传播
x_dummy = torch.randn(4, 784)  # batch=4, features=784
out = model_seq(x_dummy)
print(f"输入: {x_dummy.shape} → 输出: {out.shape}")


In [ ]:
# === 方式 2: 自定义 nn.Module（推荐，更灵活）===
class MNISTClassifier(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=[256, 128], num_classes=10, dropout=0.2):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.ReLU(),
                nn.BatchNorm1d(h_dim),
                nn.Dropout(dropout)
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model = MNISTClassifier()
print("Custom model:")
print(model)

# 参数统计
for name, param in model.named_parameters():
    print(f"  {name:30s}: {list(param.shape)}")


## 4.4 Dataset 与 DataLoader

PyTorch 的数据加载分为两层抽象：

- **`Dataset`**：定义"如何读取一个样本"。实现 `__len__` 和 `__getitem__`
- **`DataLoader`**：包装 Dataset，提供批处理、shuffle、多进程加载

### 内置数据集

`torchvision.datasets` 提供 MNIST、CIFAR-10、ImageNet 等常用数据集，自动下载。


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

# === 加载 MNIST（torchvision 内置）===
transform = transforms.Compose([
    transforms.ToTensor(),                      # PIL Image → Tensor [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 统计量标准化
])

train_dataset = datasets.MNIST(
    root='/home/yisan/ai-learning/data',
    train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root='/home/yisan/ai-learning/data',
    train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False, num_workers=0)

print(f"训练集: {len(train_dataset)} 张")
print(f"测试集: {len(test_dataset)} 张")
print(f"每批次: {len(train_loader)} 批 × 64 = {len(train_loader)*64}")

# 查看一个批次
images, labels = next(iter(train_loader))
print(f"图像 shape: {images.shape}")  # [64, 1, 28, 28]
print(f"标签: {labels[:10]}")


In [ ]:
# === 自定义 Dataset 示例 ===
class SyntheticDataset(Dataset):
    """生成合成回归数据"""
    def __init__(self, n_samples=1000, n_features=10, noise=0.1):
        self.X = torch.randn(n_samples, n_features)
        true_w = torch.randn(n_features)
        self.y = self.X @ true_w + noise * torch.randn(n_samples)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

synth_data = SyntheticDataset(n_samples=500)
synth_loader = DataLoader(synth_data, batch_size=32, shuffle=True)

x_batch, y_batch = next(iter(synth_loader))
print(f"Batch: x={x_batch.shape}, y={y_batch.shape}")


## 4.5 标准训练循环 (Training Loop)

PyTorch 训练循环遵循固定范式。理解每个步骤的作用比记忆代码更重要。

### 训练循环模板

```python
for epoch in range(num_epochs):
    # === 训练阶段 ===
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()          # 1. 清空梯度
        output = model(batch_x)        # 2. 前向传播
        loss = criterion(output, batch_y)  # 3. 计算损失
        loss.backward()                # 4. 反向传播
        optimizer.step()               # 5. 更新参数
    
    # === 评估阶段 ===
    model.eval()
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            ...
```


In [ ]:
# === 完整训练循环：MNIST MLP with PyTorch ===
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 模型、损失函数、优化器
model = MNISTClassifier(input_dim=784, hidden_dims=[128, 64], num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0
    for data, target in loader:
        data = data.view(data.size(0), -1).to(device)  # flatten 28x28 → 784
        target = target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
    
    return total_loss / len(loader), correct / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    for data, target in loader:
        data = data.view(data.size(0), -1).to(device)
        target = target.to(device)
        output = model(data)
        total_loss += criterion(output, target).item()
        correct += (output.argmax(1) == target).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

# 训练 5 epoch
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
for epoch in range(5):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Acc={train_acc:.3%} | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.3%}")

print(f"\n✅ 测试集最终准确率: {val_acc:.2%}")


## 4.6 模型保存与加载

### 两种保存方式

| 方式 | 内容 | 用途 |
|------|------|------|
| `torch.save(model.state_dict(), path)` | 仅参数 | **推荐**——体积小，灵活 |
| `torch.save(model, path)` | 完整模型 | 简单但耦合代码结构 |

`.state_dict()` 是一个 Python 字典，将每层的参数名映射到参数 Tensor：
$$\text{state_dict} = \{ \text{"layer.weight"}: \text{Tensor}, \text{"layer.bias"}: \text{Tensor}, \ldots \}$$


In [ ]:
# === 保存模型 ===
save_path = '/home/yisan/ai-learning/models/mnist_mlp.pth'
os.makedirs('/home/yisan/ai-learning/models', exist_ok=True)

# 保存 state_dict（推荐）
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': 5,
    'val_acc': val_acc,
}, save_path)
print(f"模型保存至: {save_path}")

# === 加载模型 ===
loaded_model = MNISTClassifier(input_dim=784, hidden_dims=[128, 64])
checkpoint = torch.load(save_path, map_location=device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)

# 验证加载的模型
loaded_val_loss, loaded_val_acc = evaluate(loaded_model, test_loader, criterion, device)
print(f"恢复模型验证准确率: {loaded_val_acc:.3%} (原始: {val_acc:.3%})")

# 清理
import os as _os
_os.remove(save_path)


## 本章小结

| 组件 | PyTorch 实现 |
|------|-------------|
| 多维数组 | `torch.Tensor` (CPU/GPU) |
| 自动求导 | `.backward()` + `requires_grad=True` |
| 网络层 | `nn.Linear`, `nn.Conv2d`, `nn.ReLU`... |
| 模型容器 | `nn.Sequential` / 自定义 `nn.Module` |
| 损失函数 | `nn.MSELoss`, `nn.CrossEntropyLoss`... |
| 优化器 | `optim.SGD`, `optim.Adam`... |
| 数据加载 | `Dataset` + `DataLoader` |
| 模型保存 | `torch.save(model.state_dict(), ...)` |

✅ 第五章完成！下一章将用 PyTorch 在 CIFAR-10 上构建卷积神经网络。
